# fedOPD · Exploratory Data Analysis
**Cross-silo federated forecasting of NHS outpatient (RTT) demand — operations research, not clinical.**

EDA on NHS England **Referral-to-Treatment** data: 96 monthly files, `2018-04 → 2026-03` (8 financial years).
Goal — verify the raw data, build the canonical **`[date, branch, dept, demand]`** table, and characterise it
enough to (a) select federated clients and (b) design the forecaster.

- **Grain:** trust × treatment function × month.
- **Demand:** `RTT Part Type == Part_3` (new RTT periods), column `Total All`.
- **Client:** one NHS trust (3-char `R##` provider code); independents (5-char) excluded.
- **Hierarchy:** NHS England → region → ICB (`Provider Parent`) → trust → department.

This **single notebook** runs stages 2 → 8 in sequence. Each stage ends with quantified findings (`F-xx`)
and decisions (`D-xx`) that are copied into `docs/WORKLOG.md`. Derived tables → `reports/tables/`,
figures → `reports/figures/`. Raw `data/` is read-only.

## Stage 2 · Input & inspect

**Questions this stage answers:**
(a) all 96 files exist and read · (b) the 9 canonical columns are present in every era ·
(c) the exact `Part_3` demand label · (d) trust vs independent provider-code split ·
(e) `C_999` aggregate specialty present · (f) `Period` parses (incl. the `APRIL`/`April` case quirk) ·
(g) `Total All` is numeric · (h) national totals match externally-verified figures ·
(i) schema is consistent across all 96 files.

**Sentinel months** (develop on 4; full 96-file passes only where marked): `2018-04` oldest / first modern
layout (69 cols) · `2020-04` COVID collapse · `2021-04` layout step to 121 cols · `2026-03` newest.

**Done when** `D-01` (loader `usecols` + parse rules) is logged.

In [9]:
import sys , platform , datetime as dt
import pandas as pd , numpy as np


print(dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S"), "|", platform.node())

print("username : Nilesh")

print("python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__)

2026-07-11 03:18:11 | ubuntu-Standard-PC-Q35-ICH9-2009
username : Nilesh
python 3.10.12 | pandas 2.3.3 | numpy 2.2.6


In [10]:
import os , shutil , subprocess , platform 

print("System Specifications")

cpu = platform.processor()
if shutil.which("lscpu"):                                            # lscpu gives the proper model name
    out = subprocess.run(["lscpu"], capture_output=True, text=True).stdout
    cpu = next((l.split(":",1)[1].strip() for l in out.splitlines()
                if l.startswith("Model name")), cpu)                 # grab the "Model name" line
print(f"CPU: {cpu} | {os.cpu_count()} cores")   

ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
print(f"RAM: {ram_gb:.0f} GB")

if shutil.which("nvidia-smi"):
    q = subprocess.run(["nvidia-smi",
 "--query-gpu=name,memory.total,memory.used",
 "--format=csv,noheader"], capture_output=True, text=True)
    print("GPU:" , q.stdout.strip())


System Specifications
CPU: Intel Xeon Processor (Cascadelake) | 42 cores
RAM: 67 GB
GPU: NVIDIA A100 80GB PCIe, 81920 MiB, 1 MiB


In [11]:
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "CLAUDE.md").exists())
DATA = ROOT / "data" / "nhs_rtt_referral_to_treatment"
files = sorted(DATA.glob("rtt_*.csv"))

print(len(files), "files:", files[0].name, "→", files[-1].name) 
sizes = [f.stat().st_size/1e6 for f in files]
print(f"MB: min {min(sizes):.0f} max {max(sizes):.0f}")
assert len(files) == 96  

96 files: rtt_2018-04.csv → rtt_2026-03.csv
MB: min 42 max 114


In [12]:
print("Sentinel Months placed as markers")
Sentinels = ["2018-04", "2020-04", "2021-04", "2026-03"]       # sentinels: oldest, COVID crash, mid, newest
df = {m: pd.read_csv(DATA/f"rtt_{m}.csv",              # read each sentinel into a dict of DataFrames
                     low_memory=False) for m in Sentinels}     # low_memory=False: infer dtypes from whole file
for m in Sentinels: print(m, df[m].shape)                      # rows × columns per era (expect 69 vs 121 cols)

Sentinel Months placed as markers
2018-04 (196340, 69)
2020-04 (125460, 69)
2021-04 (160251, 121)
2026-03 (183400, 121)


In [13]:
print("All columns present (schema grows 69 -> 121 across the window)")
new = df["2026-03"].columns                                   # newest era is the superset (121 cols)
old = set(df["2018-04"].columns)                              # oldest era (69 cols)
for i, c in enumerate(new, 1):
    print(f"{i:>3}. {c}" + ("" if c in old else "   (2021+ only)"))  # flag columns added later

All columns present (schema grows 69 -> 121 across the window)
  1. Period
  2. Provider Parent Org Code
  3. Provider Parent Name
  4. Provider Org Code
  5. Provider Org Name
  6. Commissioner Parent Org Code
  7. Commissioner Parent Name
  8. Commissioner Org Code
  9. Commissioner Org Name
 10. RTT Part Type
 11. RTT Part Description
 12. Treatment Function Code
 13. Treatment Function Name
 14. Gt 00 To 01 Weeks SUM 1
 15. Gt 01 To 02 Weeks SUM 1
 16. Gt 02 To 03 Weeks SUM 1
 17. Gt 03 To 04 Weeks SUM 1
 18. Gt 04 To 05 Weeks SUM 1
 19. Gt 05 To 06 Weeks SUM 1
 20. Gt 06 To 07 Weeks SUM 1
 21. Gt 07 To 08 Weeks SUM 1
 22. Gt 08 To 09 Weeks SUM 1
 23. Gt 09 To 10 Weeks SUM 1
 24. Gt 10 To 11 Weeks SUM 1
 25. Gt 11 To 12 Weeks SUM 1
 26. Gt 12 To 13 Weeks SUM 1
 27. Gt 13 To 14 Weeks SUM 1
 28. Gt 14 To 15 Weeks SUM 1
 29. Gt 15 To 16 Weeks SUM 1
 30. Gt 16 To 17 Weeks SUM 1
 31. Gt 17 To 18 Weeks SUM 1
 32. Gt 18 To 19 Weeks SUM 1
 33. Gt 19 To 20 Weeks SUM 1
 34. Gt 20 To 21 Weeks

In [14]:
print("Canonical Months that are most useful")
CANON = ["Period","Provider Org Code","Provider Org Name",        # the 9 columns the loader
         "Treatment Function Code","Treatment Function Name",     # will depend on
         "RTT Part Type","Total All"]

for m in Sentinels:
    print(m, "missing:", [c for c in CANON if c not in df[m].columns])  # any canonical col absent?
    print("  parent cols:", [c for c in df[m].columns if "Parent" in c]) # exact Provider Parent headers

Canonical Months that are most useful
2018-04 missing: []
  parent cols: ['Provider Parent Org Code', 'Provider Parent Name', 'Commissioner Parent Org Code', 'Commissioner Parent Name']
2020-04 missing: []
  parent cols: ['Provider Parent Org Code', 'Provider Parent Name', 'Commissioner Parent Org Code', 'Commissioner Parent Name']
2021-04 missing: []
  parent cols: ['Provider Parent Org Code', 'Provider Parent Name', 'Commissioner Parent Org Code', 'Commissioner Parent Name']
2026-03 missing: []
  parent cols: ['Provider Parent Org Code', 'Provider Parent Name', 'Commissioner Parent Org Code', 'Commissioner Parent Name']
